In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
train_data=pd.read_csv('/kaggle/input/llm-classification-finetuning/train.csv')
test_data=pd.read_csv('/kaggle/input/llm-classification-finetuning/test.csv')
submission=pd.read_csv('/kaggle/input/llm-classification-finetuning/sample_submission.csv')

In [ ]:
def merge_text(text):
    return f"Prompt: {train_data['prompt']} \n\n Response A: {train_data['response_a']} \n\n Response B: {train_data['response_b']}"

train_data['text']=train_data.apply(merge_text, axis=1)
test_data['text']=test_data.apply(merge_text, axis=1)

In [ ]:
def get_label(row):
    if row['winner_model_a'] == 1:
        return 0
    elif row['winner_model_b'] == 1:
        return 1
    elif row['winner_tie'] == 1:
        return 2

train_data['label'] = train_data.apply(get_label, axis=1)

In [ ]:
print(f'Total: {len(train_data)}')

In [ ]:
train_data.head(1)

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    train_data['prompt'], train_data['label'], test_size=0.2, random_state=42
)

print(f"Training set size {X_train.shape[0]}")
print(f"Validation set size {X_val.shape[0]}")

In [ ]:
!pip install transformers -U --no-cache-dir
!pip install -U transformers datasets --upgrade

In [ ]:
import transformers
import torch
print(transformers.__version__)

In [ ]:
from datasets import Dataset
train_dataset=Dataset.from_pandas(pd.DataFrame({'text': X_train, 'label':y_train}))
val_dataset=Dataset.from_pandas(pd.DataFrame({'text':X_val, 'label':y_val}))
test_dataset=Dataset.from_pandas(pd.DataFrame({'id':test_data['id'],'text':test_data['text']}))

In [ ]:
test_dataset

In [ ]:
# Tokenizer

from transformers import AutoTokenizer
model_name="bert-large-uncased"
tokenizer=AutoTokenizer.from_pretrained(model_name)

# truncation: Prevents input longer than BERT's max_length
# max_length: Forces all inputs to a fixed_length
# padding: Makes sure batches are of same size which improves compatibility and speed
# batched: ensure huggingface processes in batches

def tokenize(batch):
    return tokenizer(batch['text'], padding="max_length", truncation=True, max_length=512)

train_dataset=train_dataset.map(tokenize, batched=True)
val_dataset=val_dataset.map(tokenize, batched=True)
# test_data=test_data.map(tokenize, batched=True)
test_dataset=test_dataset.map(tokenize, batched=True)

In [ ]:
print("CUDA available:", torch.cuda.is_available())
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

In [ ]:
# Load model for classification
from transformers import AutoModelForSequenceClassification

def model_init():
    return AutoModelForSequenceClassification.from_pretrained(
        "bert-base-uncased",  # or your specific model
        num_labels=3  # adjust according to your classification task
    )

In [ ]:
!pip install optuna

In [ ]:
import optuna

def model_hp_space(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 2e-5, 5e-5, log=True),
        "num_train_epochs": trial.suggest_int("num_train_epochs", 2, 3),
        "per_device_train_batch_size": trial.suggest_categorical("per_device_train_batch_size", [8, 16]),
        "weight_decay": trial.suggest_float("weight_decay", 0.0, 0.2),
    }

def compute_objective(metrics):
   return metrics.get("eval_f1", metrics.get("eval_accuracy", 0))

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds, average='weighted'),
        'precision': precision_score(labels, preds, average='weighted'),
        'recall': recall_score(labels, preds, average='weighted'),
    }

In [ ]:
from sklearn.metrics import accuracy_score
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)
    return {'accuracy': accuracy_score(labels, preds)}

In [ ]:
from transformers import TrainingArguments, Trainer

training_args=TrainingArguments(
    output_dir="./results",
    # evaluation_strategy="epoch",
    # per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    # num_train_epochs=1.5,
    # weight_decay=0.01,
    logging_steps=300,
    fp16=False,
    bf16=True,
    save_total_limit=1,
    logging_dir='./logs',
    report_to="none",
)

small_train_dataset = train_dataset
small_val_dataset = val_dataset

trainer=Trainer(
    model_init=model_init,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_val_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
best_trial = trainer.hyperparameter_search(
    direction="maximize",
    hp_space=model_hp_space,
    compute_objective=compute_objective,
    n_trials=5,
    backend="optuna"
)

In [ ]:
# Model Training

trainer.train()

In [ ]:
print(best_trial)
print("Best learning rate:", best_trial.hyperparameters["learning_rate"])

In [ ]:
preds_output = trainer.predict(test_dataset)
predictions = np.argmax(preds_output.predictions, axis=1)

In [ ]:
submission = pd.DataFrame({
    "id": test_dataset["id"],
    "label": predictions
})
submission.to_csv("submission.csv", index=False)
print("Done")